# 05 — Explainability and stability
This notebook creates local feature-sensitivity diagnostics. It is not SHAP and does not establish causality. Use explanations for investigation, not to tune against the held-out test set.


In [ ]:
import joblib, numpy as np, pandas as pd
from src.data.loader import read_flow_csvs
from src.data.preprocessing import clean_frame
from src.explainability import local_feature_sensitivity, global_permutation_sensitivity
bundle=joblib.load('artifacts/cloudsentinel_pipeline.joblib'); prep,det=bundle['preprocessor'],bundle['detector']
raw,_=read_flow_csvs('data/raw'); X,labels=clean_frame(raw)
# This sample is for post-hoc explanation inspection only; do not use it to tune thresholds.
Xenc=prep.transform(X.sample(min(500,len(X)),random_state=42))
names=prep.get_feature_names_out()
print('encoded features',len(names),'sample rows',Xenc.shape[0])


In [ ]:
# Pick one example and inspect which one-feature median replacements change the supervised score most.
background=np.asarray(Xenc[:min(100,len(Xenc))].toarray() if hasattr(Xenc,'toarray') else Xenc[:min(100,len(Xenc))])
row=np.asarray(Xenc[0].toarray() if hasattr(Xenc[0],'toarray') else Xenc[0]).ravel()
local=local_feature_sensitivity(det.supervised,row,background,names,top_k=12)
display(local)
from pathlib import Path
out=Path('outputs/explanations'); out.mkdir(parents=True,exist_ok=True)
local.to_csv(out/'local_feature_sensitivity_example.csv',index=False)


## Interpretation guardrails
- These are one-feature perturbation sensitivities, not SHAP values and not causal explanations.
- Correlated flow features can make single-feature replacements unrealistic.
- Explanation stability/fidelity remains an experiment to perform and report; do not claim it is validated just because a table exists.
- Optional SHAP can be added for tree-specific attribution, but compare explanations with perturbation/fidelity checks.
